# 01 · Einstieg: Colab und ein IFC-Modell

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LTplus-AG/bxmid03-hs26/blob/main/notebooks/2026-10-27_Automatisierung-I/01_Einstieg_Colab_und_IFC.ipynb)

**BXMid03 · HS26 · Automatisierung I**

In diesem Notebook lernst du:
1. wie Google Colab funktioniert,
2. wie du ein IFC-Modell mit Python öffnest und erkundest,
3. wie du **eine Regel** an allen Bauteilen einer Klasse prüfst (bestanden / durchgefallen / fehlend),
4. wie du das Ergebnis als Tabelle und CSV erhältst,
5. dass dasselbe auch mit **IFC-Lite** geht.

Du musst nichts installieren. Alles läuft im Browser.

## So funktioniert Colab

| Was | Wie |
|---|---|
| **Zelle ausführen** | auf ▶ links neben der Zelle klicken oder `Umschalt + Enter` |
| **Alles ausführen** | Menü *Laufzeit → Alle ausführen* |
| **Textzellen** | erklären, was passiert (wie diese hier) |
| **Codezellen** | enthalten Python. Zeilen mit `#` sind Kommentare und werden nicht ausgeführt |
| **Reihenfolge** | Zellen bauen aufeinander auf: von oben nach unten ausführen |
| **Dateien** | Ordner-Symbol in der linken Leiste zeigt die Dateien der Sitzung |
| **Speichern** | *Datei → Kopie in Drive speichern* oder *Kopie in GitHub speichern* |

> **Wichtig:** Colab läuft auf einem Rechner von Google. Nach einer Pause (ca. 90 Min.) wird er zurückgesetzt:
> Variablen und hochgeladene Dateien sind dann weg. Einfach wieder von oben ausführen.

> **Tipp:** Das ✦-Symbol (Gemini) erklärt dir eine Zelle oder eine Fehlermeldung in Worten.

## Schritt 0 · Werkzeuge installieren

Colab hat Python schon. Die IFC-Bibliotheken holen wir mit `pip`. Das `!` am Anfang bedeutet:
„Diesen Befehl im Terminal ausführen, nicht in Python“.

- **IfcOpenShell** – die bekannteste Open-Source-Bibliothek für IFC
- **ifclite-geom** – der Python-Zugang zu IFC-Lite (gleicher Kern wie [ifclite.com](https://www.ifclite.com))

In [ ]:
#@title Installieren (dauert ca. 20 Sekunden)
!pip install -q ifcopenshell ifclite-geom
print("Fertig.")

## Schritt 1 · Modell wählen

Diese Zelle ist ein **Formular**: rechts erscheint eine Auswahlliste statt Code.
Wenn du eine andere Auswahl triffst, läuft die Zelle automatisch neu. Danach die folgenden Zellen erneut ausführen
(oder *Laufzeit → Alle ausführen*).

In [ ]:
#@title Modell wählen { run: "auto" }
#@markdown Wähle ein Beispielmodell aus der Liste. Die Zelle lädt es aus dem Kurs-Repository auf GitHub.
#@markdown „Eigene Datei hochladen“ öffnet einen Dateidialog – nur Modelle verwenden, die das Projekt verlassen dürfen.
modell = "Tragwerk · BIMcollab-Beispiel (IFC2X3)" #@param ["Tragwerk · BIMcollab-Beispiel (IFC2X3)", "Brücke · IfcBridge-Beispiel (IFC4X3)", "Hochhaus · Snowdon Towers Tragwerk (IFC2X3)", "Holzrahmenwand · fehlerhaft (IFC4)", "Holzrahmenwand · korrigiert (IFC4)", "Eigene Datei hochladen"]

import urllib.request

BASIS = "https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/main/modelle/"
DATEIEN = {
    "Tragwerk · BIMcollab-Beispiel (IFC2X3)": "Team_5_BIMcollab_Example_STR.ifc",
    "Brücke · IfcBridge-Beispiel (IFC4X3)": "Team_1_Bruecke_Arbeitsmodell.ifc",
    "Hochhaus · Snowdon Towers Tragwerk (IFC2X3)": "Team_4_Snowdon_Structural_Baufortschritt.ifc",
    "Holzrahmenwand · fehlerhaft (IFC4)": "Demo_Holzrahmen_fehlerhaft.ifc",
    "Holzrahmenwand · korrigiert (IFC4)": "Demo_Holzrahmen_korrigiert.ifc",
}

if modell == "Eigene Datei hochladen":
    from google.colab import files
    pfad = list(files.upload())[0]          # Dateidialog; die Datei liegt danach auf dem Google-Server
else:
    pfad = DATEIEN[modell]
    urllib.request.urlretrieve(BASIS + pfad, pfad)   # Datei aus GitHub herunterladen

print("Modell bereit:", pfad)

## Schritt 2 · Erster Blick ins Modell

`ifcopenshell.open()` liest die IFC-Datei. Danach fragen wir:
- Welches **Schema** (IFC2X3, IFC4, IFC4X3)?
- Welche **Bauteilklassen** kommen wie oft vor?

`IfcElement` ist die Oberklasse aller physischen Bauteile (Wände, Decken, Stützen …).

In [ ]:
#@title Modell öffnen und Bauteile zählen
import ifcopenshell
import ifcopenshell.util.element as ue     # Hilfsfunktionen: Psets, Typ, Material, Geschoss
import ifcopenshell.util.unit as uu        # Hilfsfunktionen: Einheiten
import pandas as pd                        # Tabellen

m = ifcopenshell.open(pfad)
print("Schema:", m.schema)

# Alle physischen Bauteile, ohne Öffnungen (die sind nur „Löcher“)
bauteile = [e for e in m.by_type("IfcElement") if not e.is_a("IfcOpeningElement")]
print("Bauteile:", len(bauteile))

# Wie oft kommt jede Klasse vor? value_counts() zählt und sortiert.
pd.Series([e.is_a() for e in bauteile]).value_counts().rename("Anzahl").to_frame()

## Schritt 3 · Ein Bauteil genau ansehen

Jedes Bauteil hat
- **Attribute** (fest im Schema): `GlobalId`, `Name`, …
- **Property Sets (Psets)**: Eigenschaften wie `IsExternal` oder `FireRating`
- **Quantity Sets (Qtos)**: Mengen wie `Width`, `Length`, `GrossVolume`
- oft einen **Typ** (z. B. „Wand 20 cm“), dessen Eigenschaften für alle Exemplare gelten
- ein **Geschoss** und ein **Material**

Wähle die Klasse im Formular. Gezeigt wird das erste Bauteil dieser Klasse.

In [ ]:
#@title Ein Bauteil im Detail
klasse = "IfcWall" #@param ["IfcWall", "IfcSlab", "IfcBeam", "IfcColumn", "IfcFooting", "IfcMember", "IfcBearing", "IfcDoor", "IfcWindow"]

auswahl = m.by_type(klasse)              # by_type findet auch Unterklassen, z. B. IfcWallStandardCase
if not auswahl:
    print(f"Im Modell gibt es keine {klasse}. Wähle eine andere Klasse.")
else:
    b = auswahl[0]
    print("Klasse:   ", b.is_a())
    print("GlobalId: ", b.GlobalId)
    print("Name:     ", b.Name)
    typ = ue.get_type(b)
    print("Typ:      ", typ.Name if typ else "–")
    geschoss = ue.get_container(b)
    print("Geschoss: ", geschoss.Name if geschoss else "–")
    materialien = [x.Name for x in ue.get_materials(b) if x is not None and x.Name]   # bei Schichtaufbauten mehrere
    print("Material: ", ", ".join(materialien) if materialien else "–")
    print()
    # get_psets liefert ein Wörterbuch: {Set-Name: {Eigenschaft: Wert}}. Typ-Werte sind mit drin.
    for set_name, werte in ue.get_psets(b).items():
        print(set_name)
        for name, wert in werte.items():
            if name != "id":
                print(f"   {name}: {wert}")

## Schritt 4 · Eine Regel prüfen

Eine Regel hat immer drei Teile:
1. **Auswahl** – welche Bauteile? (z. B. alle Wände)
2. **Anforderung** – was muss gelten? (z. B. Breite > 0.05 m)
3. **Ergebnis pro Bauteil** – *bestanden*, *durchgefallen* oder *fehlend* (Wert gar nicht vorhanden)

**Einheiten:** Viele Modelle speichern Längen in Millimetern. `calculate_unit_scale` liefert den Faktor
zu Metern (z. B. 0.001), damit die Regel immer in Metern formuliert werden kann.

In [ ]:
#@title Regel definieren und prüfen
klasse    = "IfcWall" #@param ["IfcWall", "IfcSlab", "IfcBeam", "IfcColumn", "IfcFooting", "IfcMember"]
groesse   = "Width"   #@param ["Width", "Height", "Length", "Depth"]
mindestwert_m = 0.05  #@param {type:"number"}

def menge(element, name):
    """Sucht eine Menge (z. B. 'Width') in allen Mengen-Sets eines Bauteils.

    Mengen stehen je nach Software in 'Qto_WallBaseQuantities', 'BaseQuantities' o. ä.
    Wir suchen deshalb in allen Sets und nehmen den ersten Treffer.
    """
    for qset in ue.get_psets(element, qtos_only=True).values():
        if name in qset:
            return qset[name]
    return None

faktor = uu.calculate_unit_scale(m)       # Länge im Modell -> Meter

zeilen = []
for b in m.by_type(klasse):
    roh = menge(b, groesse)
    if roh is None:
        ist_m, ergebnis = None, "fehlend"
    else:
        ist_m = round(roh * faktor, 4)
        ergebnis = "bestanden" if ist_m > mindestwert_m else "durchgefallen"
    zeilen.append({"GlobalId": b.GlobalId, "Klasse": b.is_a(), "Name": b.Name,
                   f"{groesse} [m]": ist_m, "Regel": f"{groesse} > {mindestwert_m} m", "Ergebnis": ergebnis})

ergebnis = pd.DataFrame(zeilen)
print(f"{len(ergebnis)} Bauteile geprüft (Einheitenfaktor {faktor})")
print(ergebnis["Ergebnis"].value_counts().to_string() if len(ergebnis) else "Keine Bauteile dieser Klasse.")

> **Achtung „grün“:** 0 geprüfte Bauteile heisst nicht „alles in Ordnung“, sondern „Auswahl leer“.
> Prüfe immer zuerst, *wie viele* Bauteile geprüft wurden.

## Schritt 5 · Ergebnis als Tabelle und CSV

Eine `DataFrame`-Tabelle lässt sich sortieren, filtern und als CSV speichern (öffnet in Excel).

In [ ]:
#@title Tabelle anzeigen (zuerst die Problemfälle)
reihenfolge = {"durchgefallen": 0, "fehlend": 1, "bestanden": 2}
ergebnis.sort_values("Ergebnis", key=lambda s: s.map(reihenfolge)).head(20)

In [ ]:
#@title Als CSV herunterladen
ergebnis.to_csv("ergebnis.csv", index=False, sep=";")   # Semikolon: öffnet direkt in Excel (CH)
from google.colab import files
files.download("ergebnis.csv")

## Schritt 6 · Dasselbe mit IFC-Lite

IFC-Lite ist eine zweite, sehr schnelle IFC-Bibliothek (Kern in Rust). `entity_data` liefert alle Bauteile
mit Attributen, Psets und Mengen als einfache Python-Daten – ohne IfcOpenShell.

Unterschied: IFC-Lite kennt keine Klassen-Vererbung. Wir nehmen deshalb alle Klassen, deren Name mit
der gewählten Klasse beginnt (z. B. `IfcWall` **und** `IfcWallStandardCase`), aber keine Typen (`…Type`).

In [ ]:
#@title Gleiche Regel mit IFC-Lite
import ifclite_geom

daten = ifclite_geom.entity_data(open(pfad, "rb").read())
faktor_lite = daten["length_unit_scale"]          # gleicher Einheitenfaktor wie oben

def passt(ifc_typ):
    return ifc_typ.startswith(klasse) and not ifc_typ.endswith("Type")

lite = {}
for e in daten["entities"].values():
    if not passt(e["ifc_type"]):
        continue
    werte = {q["name"]: q["value"] for s in e["quantity_sets"] for q in s["quantities"]}
    roh = werte.get(groesse)
    if roh is None:
        lite[e["global_id"]] = "fehlend"
    else:
        lite[e["global_id"]] = "bestanden" if roh * faktor_lite > mindestwert_m else "durchgefallen"

gleich = lite == dict(zip(ergebnis["GlobalId"], ergebnis["Ergebnis"])) if len(ergebnis) else not lite
print(f"IFC-Lite: {len(lite)} Bauteile geprüft ->", pd.Series(lite).value_counts().to_dict())
print("Gleiches Ergebnis wie IfcOpenShell:", gleich)

## Probiert selbst

1. Wähle oben ein anderes Modell und führe alles neu aus. Wie viele Bauteile sind *fehlend*? Warum?
2. Setze `mindestwert_m` auf `0.15`. Wie viele Wände fallen jetzt durch?
3. Prüfe `IfcBeam` mit `Length`. Welche Länge hat der kürzeste Träger?
4. Schau im Ordner-Symbol links: Wo liegt `ergebnis.csv`?

**Weiter:** Notebook 02 zeigt Diagramme, Filter, IDS und eine 3D-Ansicht.